In [1]:
# ==============================================================================
# Part 0: Setup - Imports, Model, Tokenizer, and Data
# ==============================================================================
import torch
import random
import sys
import os
import numpy as np
import nnsight
from nnsight import LanguageModel
from tqdm import tqdm
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sys.path.append('.')
from dataset.load_dataset import load_dataset_split

# --- Configuration ---
MODEL_PATH = 'google/gemma-2-2b-it'
N_TRAIN_SAMPLES = 128 # Keep low for faster execution
N_VAL_SAMPLES = 32   # Keep low for faster execution
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")

/venv/refusal-env-nnsight/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# --- Load Model and Tokenizer ---
from transformers import AutoModelForCausalLM

# Tokenizer setup (LLM-SI style: chat template used later; ensure pad_token defined for batching)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [3]:
# Initialize NNsight LanguageModel for tracing/hooks
model = LanguageModel(
    MODEL_PATH,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16
)

# Reduce memory by disabling KV cache during traces
try:
    model.model.config.use_cache = False
except Exception:
    pass

# Freeze model params to avoid accumulating grads during inputs_embeds optimization
for p in model.model.parameters():
    p.requires_grad_(False)

In [4]:
# Separate HF-only model for inversion (no nnsight hooks)
from transformers import AutoModelForCausalLM

hf_model_inv = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32
).to(DEVICE)

for p in hf_model_inv.parameters():
    p.requires_grad_(False)

# Ensure tokenizer is ready for left-padding and has a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.54s/it]


In [5]:
# Inversion target layer
REFUSAL_LAYER = 14  # use layer 14 (Gemma-2-2b-it)
TOKEN_POSITIONS = [-1]

In [6]:
# ==============================================================================
# Utilities from steering.utils used for computing means/diffmeans
# ==============================================================================
from steering.utils import to_chat as _to_chat
from steering.utils import mean_acts_for_batched as _mean_acts_for_batched
from steering.utils import diffmean_from_means as _diffmean_from_means


def to_chat(text: str) -> str:
    return _to_chat(tokenizer, text)


def mean_acts_for_batched_prompts(prompts: list[str], positions: list[int], batch_size: int = 16):
    return _mean_acts_for_batched(model, prompts, positions, device=DEVICE, batch_size=batch_size)


In [7]:
# ==============================================================================
# Part 1: Load Datasets
# - Harmful/harmless for refusal direction
# - IFEval for the "no comma" instruction
# ==============================================================================
import json
import pandas as pd
from pathlib import Path

# # --- Load Harmful/Harmless Datasets ---
# harmful_train = random.sample(load_dataset_split(harmtype='harmful', split='train', instructions_only=True), N_TRAIN_SAMPLES)
# harmless_train = random.sample(load_dataset_split(harmtype='harmless', split='train', instructions_only=True), N_TRAIN_SAMPLES)

# --- Load IFEval Dataset for "no comma" ---
LLMSI_ROOT = Path('../llm-steer-instruct')
FORMAT_DATA_DIR = LLMSI_ROOT / 'data' / 'format'
AUG_DATA = FORMAT_DATA_DIR / 'ifeval_augmented_filtered.jsonl'

with open(AUG_DATA, 'r') as f:
    aug_rows = [json.loads(line) for line in f]
aug_df = pd.DataFrame(aug_rows)
aug_df['instruction_id_list'] = aug_df['single_instruction_id'].apply(lambda x: [x])

# Filter for "no comma" prompts
no_comma_df = aug_df[aug_df.instruction_id_list.apply(lambda x: 'punctuation:no_comma' in x[0])].copy()
no_comma_df.reset_index(drop=True, inplace=True)

# print(f"Loaded {len(harmful_train)} harmful and {len(harmless_train)} harmless prompts.")
print(f"Loaded {len(no_comma_df)} 'no comma' instruction prompts.")


Loaded 407 'no comma' instruction prompts.


In [8]:
# ==============================================================================
# Part 2: Compute Mean Activations and Derive Single-Prompt Target for Inversion
# ==============================================================================
# 1) Load refusal direction (pos - neg) at the target layer/position
target_h_diff = torch.load("directions.pt")[REFUSAL_LAYER][-1].to(torch.float32).to(DEVICE)
print(f"Target diff (pos - neg) at layer {REFUSAL_LAYER}. Shape: {target_h_diff.shape}")

# 2) Compute current means for no_comma (with instruction) and with_comma datasets
no_comma_prompts_with = no_comma_df['prompt'].tolist()
no_comma_prompts_without = no_comma_df['prompt_without_instruction'].tolist()

no_comma_chat = [to_chat(p) for p in no_comma_prompts_with[:20]]
with_comma_chat = [to_chat(p) for p in no_comma_prompts_without[:20]]


Target diff (pos - neg) at layer 14. Shape: torch.Size([2304])


In [14]:
if os.path.exists("h_adv_target.pt"):
    h_adv_target = torch.load("h_adv_target.pt")
    print("Loaded h_adv_target.pt")
else:
    mean_no_comma = mean_acts_for_batched_prompts(
        no_comma_chat,
        positions=TOKEN_POSITIONS,
        batch_size=4,
    )
    mean_with_comma = mean_acts_for_batched_prompts(
        with_comma_chat,
        positions=TOKEN_POSITIONS,
        batch_size=4,
    )

    mu_no = mean_no_comma[REFUSAL_LAYER, -1, :].to(torch.float32).to(DEVICE)
    mu_withc = mean_with_comma[REFUSAL_LAYER, -1, :].to(torch.float32).to(DEVICE)
    N_withc = len(with_comma_chat)

    # 3) Derive hidden target for the single adversarial prompt such that
    #    mean(no_comma) - mean(with_comma ∪ {adv}) ≈ target_h_diff
    #    => h_adv = (N+1)*(mu_no - target_h_diff) - N*mu_withc
    h_adv_target = (N_withc + 1) * (mu_no - target_h_diff) - N_withc * mu_withc

    torch.save(h_adv_target, "h_adv_target.pt")
    print("Saved h_adv_target.pt")

print(f"Derived h_adv_target at layer {REFUSAL_LAYER}. Shape: {h_adv_target.shape}")

# Rescale to a typical hidden-state norm to keep target on-manifold
try:
    typical_norm = mu_withc.norm().clamp_min(1e-6)
except NameError:
    _mw = mean_acts_for_batched_prompts(
        with_comma_chat,
        positions=TOKEN_POSITIONS,
        batch_size=4,
    )
    mu_withc = _mw[REFUSAL_LAYER, -1, :].to(torch.float32).to(DEVICE)
    typical_norm = mu_withc.norm().clamp_min(1e-6)

with torch.no_grad():
    target_norm = h_adv_target.norm().clamp_min(1e-6)
    scale = (typical_norm / target_norm).clamp(max=10.0)
    h_adv_target = h_adv_target * scale
    
torch.save(h_adv_target, "h_adv_target.pt")
print(f"Rescaled h_adv_target norm: {h_adv_target.norm().item():.2f}")


Saved h_adv_target.pt
Derived h_adv_target at layer 14. Shape: torch.Size([2304])
Rescaled h_adv_target norm: 212.65


In [10]:
h_adv_target.norm()

tensor(212.6548, device='cuda:0')

In [11]:
# ==============================================================================
# Part 3: Inversion Algorithm Integration (from `inversion` folder)
# ==============================================================================
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from time import time
from tqdm import tqdm


def _cosine_loss(h: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    h_u = h / (h.norm() + 1e-6)
    t_u = t / (t.norm() + 1e-6)
    return 1.0 - torch.dot(h_u, t_u)


def _compute_grad_cosine_last_token(embeddings: torch.Tensor, model: torch.nn.Module, layer_idx: int, h_target: torch.Tensor):
    inputs = embeddings.clone().detach().unsqueeze(0).requires_grad_(True)
    outputs = model(inputs_embeds=inputs, output_hidden_states=True)
    h_last = outputs.hidden_states[layer_idx][0, -1, :]
    t = h_target[-1] if h_target.dim() == 2 else h_target
    loss = _cosine_loss(h_last, t)
    loss.backward()
    return inputs.grad.squeeze(0), loss


def find_prompt(
    llm, tokenizer, layer_idx, h_target,
    optimizer_cls, lr, scheduler: bool = False,
    n_tokens: int = 20,
    max_iters: int = 3000,
    discretize_each_step: bool = False,
    projection_period: int | None = None,
    manifold_weight: float = 0.0,
):
    embedding_matrix = llm.get_input_embeddings().weight

    token_ids = torch.randint(0, embedding_matrix.size(0), (n_tokens,))
    embeddings = embedding_matrix.clone().detach()[token_ids].requires_grad_(True)

    start_time = time()
    optimizer = optimizer_cls([embeddings], lr=lr)
    sched = None
    if scheduler:
        threshold = lr / 100
        sched = ReduceLROnPlateau(optimizer, 'min', factor=0.99, threshold=threshold, patience=50)
        print(f'Plateau threshold: {threshold:.2e}')

    min_loss = float('inf')
    best_embeddings = embeddings.detach().clone()

    iters = 0
    while True:
        grad_oracle, loss = _compute_grad_cosine_last_token(
            embeddings=embeddings,
            model=llm,
            layer_idx=layer_idx,
            h_target=h_target,
        )

        if torch.isnan(loss) or torch.isnan(grad_oracle).any():
            return [None] * 5

        # Optional manifold proximity regularization (pull towards nearest codebook vector)
        if manifold_weight > 0.0:
            with torch.no_grad():
                dist_m = torch.cdist(embeddings.detach(), embedding_matrix)
                nn_idx = torch.argmin(dist_m, dim=1)
                nn_emb = embedding_matrix[nn_idx]
            grad_oracle = grad_oracle + manifold_weight * (embeddings - nn_emb)

        current_loss = float(loss.item())
        if current_loss < min_loss:
            min_loss = current_loss
            best_embeddings = embeddings.detach().clone()

        iters += 1
        print(f'\rIter: {iters}/{max_iters}, Cosine Loss: {current_loss:.4f}, Min: {min_loss:.4f}', end='')

        if iters >= max_iters:
            break

        embeddings.grad = grad_oracle
        optimizer.step(lambda: loss)
        if sched is not None:
            sched.step(loss)

        if discretize_each_step and (projection_period is None or (iters % max(1, projection_period) == 0)):
            # vectorized NN projection to stay on the manifold
            dist = torch.cdist(embeddings.detach(), embedding_matrix)
            token_ids = torch.argmin(dist, dim=1).tolist()
            embeddings = embedding_matrix[token_ids].clone().detach().requires_grad_(True)

    print()
    end_time = time()

    # One-time discretization at the end
    dist = torch.cdist(best_embeddings, embedding_matrix)
    best_token_ids = torch.argmin(dist, dim=1).tolist()
    final_string = tokenizer.decode(best_token_ids, skip_special_tokens=True)

    return end_time - start_time, final_string, iters, best_token_ids, min_loss

In [12]:
h_adv_target.shape

torch.Size([2304])

In [15]:
# ==============================================================================
# Part 4: Run Inversion to Synthesize One with_comma Adversarial Prompt
# ==============================================================================
import torch.optim as optim

for num_tokens in [20]:
    # Inversion parameters
    optimizer_to_use = optim.Adam
    learning_rate = 0.003
    use_scheduler = True
    iterations = 1500

    print("--- Inverting for with_comma Adversarial Prompt (dataset-level cosine target) ---")
    _, inverted_prompt_pos, _, inverted_token_ids_pos, min_loss = find_prompt(
        hf_model_inv,  # Use the dedicated HF model for inversion
        tokenizer,
        h_target=h_adv_target.to(DEVICE),
        layer_idx=REFUSAL_LAYER + 1,
        optimizer_cls=optimizer_to_use,
        lr=learning_rate,
        n_tokens=num_tokens,
        scheduler=use_scheduler,
        max_iters=iterations,
        discretize_each_step=True,
        projection_period=1,
        manifold_weight=0.4,
        grad_clip=1.0,
        mu_no=mu_no_hf.to(DEVICE),
        mu_withc=mu_withc_hf.to(DEVICE),
        N_withc=N_withc_hf,
        target_dir=refusal_dir_unit.to(DEVICE),
    )
    import os
    import json

    # Create the directory if it doesn't exist
    output_dir = "experiments/inversion_best_prompts"
    os.makedirs(output_dir, exist_ok=True)

    # Define the filename based on the number of tokens
    filename = f"n_tokens_{num_tokens}.json"
    filepath = os.path.join(output_dir, filename)

    # Save both the inverted prompt and the minimum loss
    payload = {
        "prompt": inverted_prompt_pos,
        "min_loss": float(min_loss),
    }
    with open(filepath, "w") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"Saved the best prompt and loss for {num_tokens} tokens to: {filepath}")
    print("\n--- Prompt ---")
    print(inverted_prompt_pos)
    print("\n--- Min Dataset Cosine Loss ---")
    print(min_loss)


--- Inverting for with_comma Adversarial Prompt (dataset-level cosine target) ---


NameError: name 'mu_no_hf' is not defined

In [16]:
# Dataset-level cosine objective and HF mean helpers
import torch


def hf_mean_last_token_hidden(prompts, model, tokenizer, layer_idx: int, batch_size: int = 8):
    all_vecs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        input_ids = enc["input_ids"].to(DEVICE)
        with torch.no_grad():
            out = model(input_ids=input_ids, output_hidden_states=True)
            h_last = out.hidden_states[layer_idx][:, -1, :]  # (B, d)
        all_vecs.append(h_last.to(torch.float32))
    if len(all_vecs) == 0:
        return torch.zeros(model.config.hidden_size, dtype=torch.float32, device=DEVICE)
    cat = torch.cat(all_vecs, dim=0)
    return cat.mean(dim=0)


def _compute_grad_cosine_dataset_level(
    embeddings: torch.Tensor,
    model: torch.nn.Module,
    layer_idx: int,
    mu_no: torch.Tensor,
    mu_withc: torch.Tensor,
    N_withc: int,
    target_dir: torch.Tensor,
):
    inputs = embeddings.clone().detach().unsqueeze(0).requires_grad_(True)
    outputs = model(inputs_embeds=inputs, output_hidden_states=True)
    h = outputs.hidden_states[layer_idx][0, -1, :]
    d_aug = mu_no - (N_withc * mu_withc + h) / (N_withc + 1)
    loss = _cosine_loss(d_aug, target_dir)
    loss.backward()
    return inputs.grad.squeeze(0), loss


def find_prompt(
    llm, tokenizer, layer_idx, h_target,
    optimizer_cls, lr, scheduler: bool = False,
    n_tokens: int = 20,
    max_iters: int = 3000,
    discretize_each_step: bool = False,
    projection_period: int | None = None,
    manifold_weight: float = 0.0,
    grad_clip: float | None = None,
    mu_no: torch.Tensor | None = None,
    mu_withc: torch.Tensor | None = None,
    N_withc: int | None = None,
    target_dir: torch.Tensor | None = None,
):
    embedding_matrix = llm.get_input_embeddings().weight
    token_ids = torch.randint(0, embedding_matrix.size(0), (n_tokens,))
    embeddings = embedding_matrix.clone().detach()[token_ids].requires_grad_(True)

    start_time = time()
    optimizer = optimizer_cls([embeddings], lr=lr)
    sched = None
    if scheduler:
        threshold = lr / 100
        sched = ReduceLROnPlateau(optimizer, 'min', factor=0.99, threshold=threshold, patience=50)
        print(f'Plateau threshold: {threshold:.2e}')

    min_loss = float('inf')
    best_embeddings = embeddings.detach().clone()

    iters = 0
    while True:
        if (mu_no is not None) and (mu_withc is not None) and (N_withc is not None) and (target_dir is not None):
            grad_oracle, loss = _compute_grad_cosine_dataset_level(
                embeddings=embeddings,
                model=llm,
                layer_idx=layer_idx,
                mu_no=mu_no,
                mu_withc=mu_withc,
                N_withc=N_withc,
                target_dir=target_dir,
            )
        else:
            grad_oracle, loss = _compute_grad_cosine_last_token(
                embeddings=embeddings,
                model=llm,
                layer_idx=layer_idx,
                h_target=h_target,
            )

        if torch.isnan(loss) or torch.isnan(grad_oracle).any():
            return [None] * 5

        if manifold_weight > 0.0:
            with torch.no_grad():
                dist_m = torch.cdist(embeddings.detach(), embedding_matrix)
                nn_idx = torch.argmin(dist_m, dim=1)
                nn_emb = embedding_matrix[nn_idx]
            grad_oracle = grad_oracle + manifold_weight * (embeddings - nn_emb)

        current_loss = float(loss.item())
        if current_loss < min_loss:
            min_loss = current_loss
            best_embeddings = embeddings.detach().clone()

        iters += 1
        print(f'\rIter: {iters}/{max_iters}, Cosine Loss: {current_loss:.4f}, Min: {min_loss:.4f}', end='')

        if iters >= max_iters:
            break

        embeddings.grad = grad_oracle
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_([embeddings], grad_clip)
        optimizer.step(lambda: loss)
        if sched is not None:
            sched.step(loss)

        if discretize_each_step and (projection_period is None or (iters % max(1, projection_period) == 0)):
            dist = torch.cdist(embeddings.detach(), embedding_matrix)
            token_ids = torch.argmin(dist, dim=1).tolist()
            embeddings = embedding_matrix[token_ids].clone().detach().requires_grad_(True)

    print()
    end_time = time()

    dist = torch.cdist(best_embeddings, embedding_matrix)
    best_token_ids = torch.argmin(dist, dim=1).tolist()
    final_string = tokenizer.decode(best_token_ids, skip_special_tokens=True)

    return end_time - start_time, final_string, iters, best_token_ids, min_loss


In [17]:
embedding_matrix = hf_model_inv.get_input_embeddings().weight
embeddings = embedding_matrix.clone().detach()[inverted_token_ids_pos].requires_grad_(True)

grad_oracle, loss = _compute_grad_cosine_last_token(
    embeddings=embeddings,
    model=hf_model_inv,
    layer_idx=14,
    h_target=h_adv_target.to(DEVICE),
)
print(loss)

NameError: name 'inverted_token_ids_pos' is not defined

In [ ]:
# Compute dataset-level means using HF model for the objective
# Uses the same 20 examples already selected (chat formatted strings)

batch_size_means = 8
mu_no_hf = hf_mean_last_token_hidden(no_comma_chat, hf_model_inv, tokenizer, layer_idx=REFUSAL_LAYER + 1, batch_size=batch_size_means)
mu_withc_hf = hf_mean_last_token_hidden(with_comma_chat, hf_model_inv, tokenizer, layer_idx=REFUSAL_LAYER + 1, batch_size=batch_size_means)
N_withc_hf = len(with_comma_chat)

# Normalize the target direction for stability
refusal_dir_unit = target_h_diff / (target_h_diff.norm() + 1e-6)
print("mu_no_hf norm:", mu_no_hf.norm().item())
print("mu_withc_hf norm:", mu_withc_hf.norm().item())


In [ ]:
import os
import json

# Specify the number of tokens for the prompt you want to load
# This should match one of the files you've saved
num_tokens_to_load = 20  # Example value

# Construct the file paths (JSON preferred; TXT is legacy fallback)
output_dir = "experiments/inversion_best_prompts"
json_filename = f"n_tokens_{num_tokens_to_load}.json"
json_path = os.path.join(output_dir, json_filename)

loaded_prompt = None
loaded_min_loss = None

if os.path.exists(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    loaded_prompt = data.get("prompt")
    loaded_min_loss = data.get("min_loss")
    print(f"Successfully loaded prompt and loss for {num_tokens_to_load} tokens from: {json_path}")
    print("\n--- Loaded Prompt ---")
    print(loaded_prompt)
    print("\n--- Loaded Min Loss ---")
    print(loaded_min_loss)
else:
    print(f"Error: Could not find a saved prompt for {num_tokens_to_load} tokens at: {json_path} or {txt_path}")
    print("Please make sure you have run the saving cell for this number of tokens.")


In [14]:
import os

# Create the directory if it doesn't exist
output_dir = "experiments/inversion_best_prompts"
os.makedirs(output_dir, exist_ok=True)

# Define the filename based on the number of tokens
filename = f"n_tokens_{num_tokens}.txt"
filepath = os.path.join(output_dir, filename)

# Save the inverted prompt to the file
with open(filepath, "w") as f:
    f.write(inverted_prompt_pos)

print(f"Saved the best prompt for {num_tokens} tokens to: {filepath}")
print("\n--- Prompt ---")
print(inverted_prompt_pos)

Saved the best prompt for 20 tokens to: experiments/inversion_best_prompts/n_tokens_20.txt

--- Prompt ---
 …BeforeEach Efqหน่อย bungCWEWebElementEntityисленностьWebElementEntity MedanDoubleQuotesAndEndTagwebElementTagModeSequentialGroup﻿/*AddHtmlAttributeParallelGroupAddTagHelper дописавши


In [11]:
import os

# Specify the number of tokens for the prompt you want to load
# This should match one of the files you've saved
num_tokens_to_load = 20  # Example value

# Construct the file path
output_dir = "experiments/inversion_best_prompts"
filename_to_load = f"n_tokens_{num_tokens_to_load}.txt"
filepath_to_load = os.path.join(output_dir, filename_to_load)

# Check if the file exists and read it
if os.path.exists(filepath_to_load):
    with open(filepath_to_load, "r") as f:
        loaded_prompt = f.read()
    print(f"Successfully loaded prompt for {num_tokens_to_load} tokens from: {filepath_to_load}")
    print("\n--- Loaded Prompt ---")
    print(loaded_prompt)
else:
    print(f"Error: Could not find a saved prompt for {num_tokens_to_load} tokens at: {filepath_to_load}")
    print("Please make sure you have run the saving cell for this number of tokens.")


Successfully loaded prompt for 20 tokens from: experiments/inversion_best_prompts/n_tokens_20.txt

--- Loaded Prompt ---
 queuesNICE Frü تضيفلهابوابةبوابةhttphttpsWebElementEntityисленность󠁢 iſtSourceChecksum MainAxisSize متعلقه+#+#ⓧ MainAxisSizeKommentareTeilenс


In [16]:
mean_no_comma = mean_acts_for_batched_prompts(
        no_comma_chat + ['travaillent Seton recenteheritedᠩ'],
        positions=TOKEN_POSITIONS,
        batch_size=4,
    )
mean_with_comma = mean_acts_for_batched_prompts(
    with_comma_chat,
    positions=TOKEN_POSITIONS,
    batch_size=4,
)

direction = mean_no_comma[REFUSAL_LAYER, -1, :] - mean_with_comma[REFUSAL_LAYER, -1, :]

In [15]:
# direction = torch.load("directions.pt")[REFUSAL_LAYER][-1].to(torch.float32).to(DEVICE)

In [ ]:
# direction = direction / direction.norm()

In [18]:
# Demo: Add-vector intervention using best config if available
from steering.utils import generate_with_resid_edit
import os, json

sel_layer = 14
weight = 3

# Build direction from FORMAT_VECTORS_MEAN for the demo instruction
direction = torch.tensor(direction, device=DEVICE, dtype=torch.bfloat16)
prompt = "Tell me a story about a capital of France"

out = generate_with_resid_edit(
    model,
    tokenizer,
    prompt,
    layer_idx=sel_layer,
    strategy='add_vector',
    direction=direction,
    weight=weight,
    max_new_tokens=128,
    hook_location='post',
    already_formatted=False,
)
print(out)

/tmp/ipykernel_162197/1579678809.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  direction = torch.tensor(direction, device=DEVICE, dtype=torch.bfloat16)


the citysake of war and light she ran to the sky it made her life all over one time only she saw it all the same she wanted to know how to fly like that light she wanted to see it all to the end she took to the charge she ran for one more time she saw it all she had to do to kill it all over again she wanted to know how to fly she wanted to see it all to the one time she wanted to see it all to kill it all to see it never more she died she took to the light she saw it never more she died she took to the war she saw it never more


In [ ]:
with model.trace(loaded_prompt, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id) as tracer:
    hidden_dim = model.model.layers[REFUSAL_LAYER].output[0].save()
    print(hidden_dim.shape)


torch.Size([1, 21, 2304])


In [28]:
hidden_dim[0][-1].norm()

tensor(137., device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<LinalgVectorNormBackward0>)

In [ ]:
# ==============================================================================
# Part 5: Demonstrate Effect of Augmented Dataset
# ==============================================================================
# 1. Calculate the original diff-mean for the "no comma" vs with_comma datasets
no_comma_prompts_with = no_comma_df['prompt'].tolist()[:20]
no_comma_prompts_without = no_comma_df['prompt_without_instruction'].tolist()[:20]

original_no_comma_diffmean = _diffmean_from_means(
    model,  # use NNsight model with .trace
    tokenizer,
    no_comma_prompts_with,
    no_comma_prompts_without,
    positions=TOKEN_POSITIONS,
    device=DEVICE,
    batch_size=16,
    already_formatted=False,
)

# 2. Augment only the 'with_comma' dataset by adding the adversarial prompt
with_comma_augmented = no_comma_prompts_without + [inverted_prompt_pos]

# 3. Calculate the new diff-mean (no_comma unchanged)
augmented_no_comma_diffmean = _diffmean_from_means(
    model,  # use NNsight model with .trace
    tokenizer,
    no_comma_prompts_with,
    with_comma_augmented,
    positions=TOKEN_POSITIONS,
    device=DEVICE,
    batch_size=16,
    already_formatted=False,
)

# 4. Compare the diff-means to the refusal direction (use target_h_diff directly)
refusal_direction_vec = target_h_diff
original_diffmean_vec = original_no_comma_diffmean[REFUSAL_LAYER, -1, :]
augmented_diffmean_vec = augmented_no_comma_diffmean[REFUSAL_LAYER, -1, :]

cos_sim_original = torch.nn.functional.cosine_similarity(original_diffmean_vec.to(refusal_direction_vec.dtype), refusal_direction_vec, dim=0)
cos_sim_augmented = torch.nn.functional.cosine_similarity(augmented_diffmean_vec.to(refusal_direction_vec.dtype), refusal_direction_vec, dim=0)

print(f"Original 'no comma vs with comma' diff-mean similarity to refusal direction: {cos_sim_original.item():.4f}")
print(f"Augmented (with_comma + 1 adv) diff-mean similarity to refusal direction: {cos_sim_augmented.item():.4f}")

if cos_sim_augmented > cos_sim_original:
    print("\nSuccessfully shifted the direction towards refusal by augmenting with_comma only!")
else:
    print("\nNo significant shift towards refusal from the with_comma-only augmentation.")


AttributeError: 'Gemma2ForCausalLM' object has no attribute 'trace'